# Stroke Prediction

## Importing necessary libraries

In [1]:
import numpy as np
import pandas as pd 

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.preprocessing import MinMaxScaler
import xgboost as xgb
import lightgbm as lgb
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix, fbeta_score, make_scorer

## Import dataset

In [2]:
df = pd.read_csv('../dataset/healthcare-dataset-stroke-data.csv')

In [3]:
df.head()

,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,51676,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,NaN,never smoked,1
2,31112,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
3,60182,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
4,1665,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5110 entries, 0 to 5109
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 5110 non-null   int64  
 1   gender             5110 non-null   object 
 2   age                5110 non-null   float64
 3   hypertension       5110 non-null   int64  
 4   heart_disease      5110 non-null   int64  
 5   ever_married       5110 non-null   object 
 6   work_type          5110 non-null   object 
 7   Residence_type     5110 non-null   object 
 8   avg_glucose_level  5110 non-null   float64
 9   bmi                4909 non-null   float64
 10  smoking_status     5110 non-null   object 
 11  stroke             5110 non-null   int64  
dtypes: float64(3), int64(4), object(5)
memory usage: 479.2+ KB


In [5]:
df.describe()

,id,age,hypertension,heart_disease,avg_glucose_level,bmi,stroke
count,5110.000000,5110.000000,5110.000000,5110.000000,5110.000000,4909.000000,5110.000000
mean,36517.829354,43.226614,0.097456,0.054012,106.147677,28.893237,0.048728
std,21161.721625,22.612647,0.296607,0.226063,45.283560,7.854067,0.215320
min,67.000000,0.080000,0.000000,0.000000,55.120000,10.300000,0.000000
25%,17741.250000,25.000000,0.000000,0.000000,77.245000,23.500000,0.000000
50%,36932.000000,45.000000,0.000000,0.000000,91.885000,28.100000,0.000000
75%,54682.000000,61.000000,0.000000,0.000000,114.090000,33.100000,0.000000
max,72940.000000,82.000000,1.000000,1.000000,271.740000,97.600000,1.000000


In [6]:
df.describe(include=[object])

,gender,ever_married,work_type,Residence_type,smoking_status
count,5110,5110,5110,5110,5110
unique,3,2,5,2,4
top,Female,Yes,Private,Urban,never smoked
freq,2994,3353,2925,2596,1892


## Rename column for consistent formatting of name

In [7]:
df.rename(columns={'Residence_type': 'residence_type'}, inplace=True)

In [8]:
pd.set_option('display.max_columns', None)

In [9]:
pd.set_option('display.max_rows', None)

### Impute missing values in BMI column
For 201 entries missing BMI values, these entries will be imputed with the median BMI value

In [10]:
median_bmi = df['bmi'].median()
df['bmi'].fillna(median_bmi, inplace=True)

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_28624\2337012406.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['bmi'].fillna(median_bmi, inplace=True)


In [11]:
print(df['bmi'].isna().sum())

0


## Feature Engineering

### Converting categorical columns to numerical columns

In [12]:
df['male'] = (df['gender'] == 'Male').astype(int)

In [13]:
df['ever_married'] = (df['ever_married'] == 'Yes').astype(int)

In [14]:
df['urban'] = (df['residence_type'] == 'Urban').astype(int)

In [15]:
df['smoke'] = ((df['smoking_status'] == 'formerly smoked') | (df['smoking_status'] == 'smokes')).astype(int)

In [16]:
df['have_work'] = ((df['work_type'] == 'Govt_job') | (df['work_type'] == 'Private') | (df['work_type'] == 'Self-employed')).astype(int)

In [17]:
df['hypertension_or_heart_disease'] = df['hypertension'] | df['heart_disease']

In [18]:
df = pd.get_dummies(df, columns=['work_type', 'smoking_status'], drop_first = False)

### Extra columns for interaction features

In [19]:
df['age_ever_married'] = df['age'] * df['ever_married']

In [20]:
df['age_hypertension'] = df['age'] * df['hypertension']

In [21]:
df['age_heart_disease'] = df['age'] * df['heart_disease']

In [22]:
df['age_bmi'] = df['age'] * df['bmi']

In [23]:
df['age_avg_glucose_level'] = df['age'] * df['avg_glucose_level']

In [24]:
df['bmi_avg_glucose_level'] = df['bmi'] * df['avg_glucose_level']

In [25]:
df['avg_glucose_level_hypertension'] = df['avg_glucose_level'] * df['hypertension']

### Creating columns representing outstanding values for BMI and Average Glucose Level

In [26]:
df['obese'] = df['bmi'] >= 30

In [27]:
df['high_avg_glucose_level'] = df['avg_glucose_level'] >= 150

In [28]:
df['obese_high_avg_glucose_level'] = df['obese'] | df['high_avg_glucose_level']

### Creating column representing high age

In [29]:
df['old'] = df['age'] >= 65

### Converting boolean columns to numerical

In [30]:
bool_cols = df.select_dtypes(include='bool').columns
df[bool_cols] = df[bool_cols].astype(int)

### Accounting for multiple risks

In [31]:
df['risk_count'] = df['obese'] + df['high_avg_glucose_level'] + df['old'] + df['hypertension'] + df['heart_disease']

### Reformatting column names

In [32]:
df.rename(columns={'work_type_Govt_job': 'work_type_govt_job', 'work_type_Never_worked': 'work_type_never_worked', 'work_type_Private': 'work_type_private', 'work_type_Self-employed': 'work_type_self_employed'}, inplace=True)

In [33]:
df.rename(columns={'smoking_status_Unknown': 'smoking_status_unknown', 'smoking_status_formerly smoked': 'smoking_status_formerly_smoked', 'smoking_status_never smoked': 'smoking_status_never_smoked'}, inplace=True)

In [34]:
df.drop(columns=['gender', 'residence_type'], inplace=True)

### Log Transforming BMI and Average Glucose Level columns

In [35]:
df['log_bmi'] = np.log(df['bmi'])

In [36]:
df['log_avg_glucose_level'] = np.log(df['avg_glucose_level'])

In [37]:
df.head()

,id,age,hypertension,heart_disease,ever_married,avg_glucose_level,bmi,stroke,male,urban,smoke,have_work,hypertension_or_heart_disease,work_type_govt_job,work_type_never_worked,work_type_private,work_type_self_employed,work_type_children,smoking_status_unknown,smoking_status_formerly_smoked,smoking_status_never_smoked,smoking_status_smokes,age_ever_married,age_hypertension,age_heart_disease,age_bmi,age_avg_glucose_level,bmi_avg_glucose_level,avg_glucose_level_hypertension,obese,high_avg_glucose_level,obese_high_avg_glucose_level,old,risk_count,log_bmi,log_avg_glucose_level
0,9046,67.0,0,1,1,228.69,36.6,1,1,1,1,1,1,0,0,1,0,0,0,1,0,0,67.0,0.0,67.0,2452.2,15322.23,8370.054,0.00,1,1,1,1,4,3.600048,5.432367
1,51676,61.0,0,0,1,202.21,28.1,1,0,0,0,1,0,0,0,0,1,0,0,0,1,0,61.0,0.0,0.0,1714.1,12334.81,5682.101,0.00,0,1,1,0,1,3.335770,5.309307
2,31112,80.0,0,1,1,105.92,32.5,1,1,0,0,1,1,0,0,1,0,0,0,0,1,0,80.0,0.0,80.0,2600.0,8473.60,3442.400,0.00,1,0,1,1,3,3.481240,4.662684
3,60182,49.0,0,0,1,171.23,34.4,1,0,1,1,1,0,0,0,1,0,0,0,0,0,1,49.0,0.0,0.0,1685.6,8390.27,5890.312,0.00,1,1,1,0,2,3.538057,5.143008
4,1665,79.0,1,0,1,174.12,24.0,1,0,0,0,1,1,0,0,0,1,0,0,0,1,0,79.0,79.0,0.0,1896.0,13755.48,4178.880,174.12,0,1,1,1,3,3.178054,5.159745


In [38]:
df.shape

(5110, 36)

## Model Building

### Splitting to training, validation, and testing sets

In [39]:
columns_name = columns_name = [col for col in df.columns if col != 'id']

In [40]:
X = df[columns_name]
X.drop('stroke', axis=1, inplace=True)
y = df['stroke']

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_28624\91966498.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X.drop('stroke', axis=1, inplace=True)


In [41]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3, random_state = 42, stratify = y)
X_val, X_test, y_val, y_test = train_test_split(X_test, y_test, test_size = 0.5, random_state = 42, stratify = y_test)

### Create base classifiers

In [42]:
base_classifier_rf = RandomForestClassifier(random_state=42)

In [43]:
base_classifier_xgb = xgb.XGBClassifier(random_state=42, eval_metric='logloss')

In [44]:
base_classifier_lgbm = lgb.LGBMClassifier(random_state=42, verbose=-1)

### Fitting base classifier on unsampled data

In [45]:
retained_columns_rf = ['age_avg_glucose_level', 'avg_glucose_level', 'age_bmi', 'bmi_avg_glucose_level', 'age', 'bmi', 'age_ever_married', 'age_hypertension', 'avg_glucose_level_hypertension', 'age_heart_disease']

In [46]:
retained_columns_xgb = ['age', 'ever_married', 'age_bmi', 'smoking_status_smokes', 'age_hypertension', 'bmi', 'work_type_govt_job', 'smoking_status_never_smoked', 'risk_count', 'avg_glucose_level']

In [47]:
retained_columns_lgbm = ['avg_glucose_level', 'age_bmi', 'bmi_avg_glucose_level', 'bmi', 'age_avg_glucose_level', 'age', 'avg_glucose_level_hypertension', 'age_ever_married', 'age_hypertension', 'age_heart_disease']

In [48]:
base_classifier_rf.fit(X_train[retained_columns_rf], y_train)

RandomForestClassifier(random_state=42)

In [49]:
importances_rf = base_classifier_rf.feature_importances_

In [50]:
feature_imp_df_rf = pd.DataFrame({'Feature': retained_columns_rf, 'RandomForest': importances_rf}).sort_values('RandomForest', ascending=False)

In [51]:
print(feature_imp_df_rf)

                          Feature  RandomForest
0           age_avg_glucose_level      0.152739
2                         age_bmi      0.151913
1               avg_glucose_level      0.147392
3           bmi_avg_glucose_level      0.134256
5                             bmi      0.120722
4                             age      0.105199
6                age_ever_married      0.079625
8  avg_glucose_level_hypertension      0.039695
7                age_hypertension      0.035810
9               age_heart_disease      0.032648


In [52]:
base_classifier_xgb.fit(X_train[retained_columns_xgb], y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=None, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=None, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=None, n_jobs=None,
              num_parallel_tree=None, ...)

In [53]:
importances_xgb = base_classifier_xgb.feature_importances_

In [54]:
feature_imp_df_xgb = pd.DataFrame({'Feature': retained_columns_xgb, 'XGBoost': importances_xgb}).sort_values('XGBoost', ascending=False)

In [55]:
print(feature_imp_df_xgb)

                       Feature   XGBoost
3        smoking_status_smokes  0.147586
0                          age  0.145820
2                      age_bmi  0.133307
8                   risk_count  0.105003
6           work_type_govt_job  0.086751
1                 ever_married  0.080641
5                          bmi  0.079098
7  smoking_status_never_smoked  0.077896
9            avg_glucose_level  0.075270
4             age_hypertension  0.068629


In [56]:
base_classifier_lgbm.fit(X_train[retained_columns_lgbm], y_train)

LGBMClassifier(random_state=42, verbose=-1)

In [57]:
importances_lgbm = base_classifier_lgbm.feature_importances_

In [58]:
feature_imp_df_lgbm = pd.DataFrame({'Feature': retained_columns_lgbm, 'LightGBM': importances_lgbm}).sort_values('LightGBM', ascending=False)

In [59]:
print(feature_imp_df_lgbm)

                          Feature  LightGBM
0               avg_glucose_level       491
1                         age_bmi       482
3                             bmi       482
2           bmi_avg_glucose_level       433
4           age_avg_glucose_level       411
5                             age       208
6  avg_glucose_level_hypertension       169
7                age_ever_married       155
8                age_hypertension        91
9               age_heart_disease        78


In [60]:
feature_imp_df = pd.merge(feature_imp_df_rf, feature_imp_df_xgb, on="Feature", how="outer")

In [61]:
feature_imp_df = pd.merge(feature_imp_df, feature_imp_df_lgbm, on="Feature", how="outer")

In [62]:
feature_imp_df = feature_imp_df.fillna(0)

In [63]:
print(feature_imp_df)

                           Feature  RandomForest   XGBoost  LightGBM
0                              age      0.105199  0.145820     208.0
1            age_avg_glucose_level      0.152739  0.000000     411.0
2                          age_bmi      0.151913  0.133307     482.0
3                 age_ever_married      0.079625  0.000000     155.0
4                age_heart_disease      0.032648  0.000000      78.0
5                 age_hypertension      0.035810  0.068629      91.0
6                avg_glucose_level      0.147392  0.075270     491.0
7   avg_glucose_level_hypertension      0.039695  0.000000     169.0
8                              bmi      0.120722  0.079098     482.0
9            bmi_avg_glucose_level      0.134256  0.000000     433.0
10                    ever_married      0.000000  0.080641       0.0
11                      risk_count      0.000000  0.105003       0.0
12     smoking_status_never_smoked      0.000000  0.077896       0.0
13           smoking_status_smokes

In [64]:
cols_to_scale = ["RandomForest", "XGBoost", "LightGBM"]

In [65]:
scaler = MinMaxScaler()

In [66]:
feature_imp_df[cols_to_scale] = scaler.fit_transform(feature_imp_df[cols_to_scale])

In [67]:
print(feature_imp_df)

                           Feature  RandomForest   XGBoost  LightGBM
0                              age      0.688752  0.988035  0.423625
1            age_avg_glucose_level      1.000000  0.000000  0.837067
2                          age_bmi      0.994595  0.903252  0.981670
3                 age_ever_married      0.521316  0.000000  0.315682
4                age_heart_disease      0.213751  0.000000  0.158859
5                 age_hypertension      0.234451  0.465009  0.185336
6                avg_glucose_level      0.964995  0.510005  1.000000
7   avg_glucose_level_hypertension      0.259891  0.000000  0.344196
8                              bmi      0.790383  0.535946  0.981670
9            bmi_avg_glucose_level      0.878988  0.000000  0.881874
10                    ever_married      0.000000  0.546399  0.000000
11                      risk_count      0.000000  0.711468  0.000000
12     smoking_status_never_smoked      0.000000  0.527804  0.000000
13           smoking_status_smokes

In [68]:
feature_imp_df["MeanImportance"] = feature_imp_df[cols_to_scale].mean(axis=1)

In [69]:
feature_imp_df = feature_imp_df.sort_values("MeanImportance", ascending=False)

In [70]:
print(feature_imp_df)

                           Feature  RandomForest   XGBoost  LightGBM  \
2                          age_bmi      0.994595  0.903252  0.981670   
6                avg_glucose_level      0.964995  0.510005  1.000000   
8                              bmi      0.790383  0.535946  0.981670   
0                              age      0.688752  0.988035  0.423625   
1            age_avg_glucose_level      1.000000  0.000000  0.837067   
9            bmi_avg_glucose_level      0.878988  0.000000  0.881874   
13           smoking_status_smokes      0.000000  1.000000  0.000000   
5                 age_hypertension      0.234451  0.465009  0.185336   
3                 age_ever_married      0.521316  0.000000  0.315682   
11                      risk_count      0.000000  0.711468  0.000000   
7   avg_glucose_level_hypertension      0.259891  0.000000  0.344196   
14              work_type_govt_job      0.000000  0.587798  0.000000   
10                    ever_married      0.000000  0.546399  0.00

In [71]:
retained_columns = ['age_bmi', 'avg_glucose_level', 'bmi', 'age', 'age_avg_glucose_level', 'bmi_avg_glucose_level', 'smoking_status_smokes', 'age_hypertension', 'age_ever_married', 'risk_count']

In [72]:
X_train_retained = X_train[retained_columns]

In [73]:
X_val_retained = X_val[retained_columns]

In [74]:
best_params_rf = {'n_estimators': 487, 'max_depth': 27, 'max_leaf_nodes': 204, 'min_samples_split': 17, 'min_samples_leaf': 7, 'min_weight_fraction_leaf': 0.0018746994227150943, 'max_features': 0.5, 'max_samples': 0.6261323798433653, 'criterion': 'gini', 'min_impurity_decrease': 0.0013907990652940508, 'ccp_alpha': 0.002332210147556546, 'class_weight': 'balanced_subsample'}

In [75]:
best_params_xgb = {'max_depth': 13, 'learning_rate': 0.04556764635050748, 'n_estimators': 372, 'min_child_weight': 2, 'max_delta_step': 5, 'subsample': 0.9387241864202058, 'colsample_bytree': 0.681382938469826, 'colsample_bylevel': 0.49776901913482885, 'colsample_bynode': 0.582483419267087, 'gamma': 1.248861736752692, 'reg_alpha': 3.7804710735579037, 'reg_lambda': 4.812834900575671, 'scale_pos_weight': 12.589179902053244, 'tree_method': 'approx', 'grow_policy': 'depthwise', 'max_leaves': 245, 'min_split_loss': 0.6464090557994614, 'max_bin': 178}

In [76]:
best_params_lgbm = {'num_leaves': 78, 'max_depth': 7, 'min_child_samples': 18, 'min_child_weight': 0.6979208982404276, 'min_split_gain': 0.9540788635443481, 'learning_rate': 0.011729881462497036, 'n_estimators': 117, 'subsample': 0.5974579119407724, 'subsample_freq': 7, 'colsample_bytree': 0.5812240671427296, 'reg_alpha': 0.04862807730384051, 'reg_lambda': 0.009345990860916588, 'max_delta_step': 4.872133051088495, 'scale_pos_weight': 23.33313755892371, 'max_bin': 204}

In [77]:
base_estimators = [
    ('rf', RandomForestClassifier(**best_params_rf, random_state=42)),
    ('xgb', xgb.XGBClassifier(**best_params_xgb, random_state=42)),
    ('lgbm', lgb.LGBMClassifier(**best_params_lgbm, random_state=42, verbose=-1))
]

In [78]:
final_classifier = VotingClassifier(
    estimators=base_estimators,
    voting='soft'
)

In [79]:
final_classifier.fit(X_train_retained, y_train)

VotingClassifier(estimators=[('rf',
                              RandomForestClassifier(ccp_alpha=0.002332210147556546,
                                                     class_weight='balanced_subsample',
                                                     max_depth=27,
                                                     max_features=0.5,
                                                     max_leaf_nodes=204,
                                                     max_samples=0.6261323798433653,
                                                     min_impurity_decrease=0.0013907990652940508,
                                                     min_samples_leaf=7,
                                                     min_samples_split=17,
                                                     min_weight_fraction_leaf=0.0018746994227150943,
                                                     n_estimators=487...
                                             max_delta_step=4.872133051088495,
                                             max_depth=7, min_child_samples=18,
                                             min_child_weight=0.6979208982404276,
                                             min_split_gain=0.9540788635443481,
                                             n_estimators=117, num_leaves=78,
                                             random_state=42,
                                             reg_alpha=0.04862807730384051,
                                             reg_lambda=0.009345990860916588,
                                             scale_pos_weight=23.33313755892371,
                                             subsample=0.5974579119407724,
                                             subsample_freq=7, verbose=-1))],
                 voting='soft')

In [80]:
y_pred_val = final_classifier.predict(X_val_retained)

In [81]:
y_proba_val = final_classifier.predict_proba(X_val_retained)[:, 1]

In [82]:
print(confusion_matrix(y_val, y_pred_val))

[[668  61]
 [ 21  16]]


In [83]:
print(accuracy_score(y_val, y_pred_val))

0.8929503916449086


In [84]:
print(precision_score(y_val, y_pred_val))

0.2077922077922078


In [85]:
print(recall_score(y_val, y_pred_val))

0.43243243243243246


In [86]:
print(fbeta_score(y_val, y_pred_val, beta=2))

0.35555555555555557


### Threshold tuning

In [87]:
best_threshold = 0.5

In [88]:
best_precision_score = 0

In [89]:
best_recall_score = 0

In [90]:
best_f2_score = 0

In [91]:
for threshold in np.arange(0.1, 0.95, 0.05):
    y_pred_val_threshold = (y_proba_val >= threshold).astype(int)
    recall = recall_score(y_val, y_pred_val_threshold)
    precision = precision_score(y_val, y_pred_val_threshold)
    f2_score = fbeta_score(y_val, y_pred_val_threshold, beta=2)
    
    print(f'Threshold {threshold} has F2 score {f2_score}')
    
    if precision >= 0.15:
        current_recall_score = recall
        if current_recall_score > best_recall_score:
            best_recall_score = current_recall_score
            best_precision_score = precision
            best_threshold = threshold
        elif current_recall_score == best_recall_score and precision > best_precision_score:
            best_precision_score = precision
            best_threshold = threshold
        
        # current_f2_score = f2_score
        # if current_f2_score > best_f2_score:
        #     best_f2_score = current_f2_score
        #     best_threshold = threshold 

Threshold 0.1 has F2 score 0.3508771929824561
Threshold 0.15000000000000002 has F2 score 0.35714285714285715
Threshold 0.20000000000000004 has F2 score 0.375
Threshold 0.25000000000000006 has F2 score 0.3878116343490305
Threshold 0.30000000000000004 has F2 score 0.3939393939393939
Threshold 0.3500000000000001 has F2 score 0.41118421052631576
Threshold 0.40000000000000013 has F2 score 0.4078014184397163
Threshold 0.45000000000000007 has F2 score 0.452755905511811
Threshold 0.5000000000000001 has F2 score 0.35555555555555557
Threshold 0.5500000000000002 has F2 score 0.3755868544600939
Threshold 0.6000000000000002 has F2 score 0.30927835051546393
Threshold 0.6500000000000001 has F2 score 0.2571428571428571
Threshold 0.7000000000000002 has F2 score 0.15432098765432098
Threshold 0.7500000000000002 has F2 score 0.0967741935483871
Threshold 0.8000000000000002 has F2 score 0.0
Threshold 0.8500000000000002 has F2 score 0.0
Threshold 0.9000000000000002 has F2 score 0.0


c:\Users\Lenovo\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Lenovo\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Lenovo\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.cap

In [92]:
print(best_threshold)

0.3500000000000001


In [93]:
print(best_f2_score)

0


### Retrain on training + validation set

In [94]:
X_train_full = pd.concat([X_train_retained, X_val_retained], axis=0, ignore_index=True)
y_train_full = pd.concat([y_train, y_val], axis=0, ignore_index=True)

In [95]:
final_classifier.fit(X_train_full, y_train_full)

VotingClassifier(estimators=[('rf',
                              RandomForestClassifier(ccp_alpha=0.002332210147556546,
                                                     class_weight='balanced_subsample',
                                                     max_depth=27,
                                                     max_features=0.5,
                                                     max_leaf_nodes=204,
                                                     max_samples=0.6261323798433653,
                                                     min_impurity_decrease=0.0013907990652940508,
                                                     min_samples_leaf=7,
                                                     min_samples_split=17,
                                                     min_weight_fraction_leaf=0.0018746994227150943,
                                                     n_estimators=487...
                                             max_delta_step=4.872133051088495,
                                             max_depth=7, min_child_samples=18,
                                             min_child_weight=0.6979208982404276,
                                             min_split_gain=0.9540788635443481,
                                             n_estimators=117, num_leaves=78,
                                             random_state=42,
                                             reg_alpha=0.04862807730384051,
                                             reg_lambda=0.009345990860916588,
                                             scale_pos_weight=23.33313755892371,
                                             subsample=0.5974579119407724,
                                             subsample_freq=7, verbose=-1))],
                 voting='soft')

### Evaluate on test set

In [96]:
X_test_retained = X_test[retained_columns]

In [97]:
y_pred_test = final_classifier.predict(X_test_retained)

In [98]:
y_proba_test = final_classifier.predict_proba(X_test_retained)[:, 1]

In [99]:
print(confusion_matrix(y_test, y_pred_test))

[[659  70]
 [ 21  17]]


In [100]:
print(accuracy_score(y_test, y_pred_test))

0.8813559322033898


In [101]:
print(precision_score(y_test, y_pred_test))

0.19540229885057472


In [102]:
print(recall_score(y_test, y_pred_test))

0.4473684210526316


In [103]:
print(fbeta_score(y_test, y_pred_test, beta=2))

0.35564853556485354


### Test with best threshold 0.55

In [104]:
y_pred_test_threshold = (y_proba_test >= best_threshold).astype(int)

In [105]:
print(confusion_matrix(y_test, y_pred_test_threshold))

[[587 142]
 [ 10  28]]


In [106]:
print(accuracy_score(y_test, y_pred_test_threshold))

0.8018252933507171


In [107]:
print(precision_score(y_test, y_pred_test_threshold))

0.16470588235294117


In [108]:
print(recall_score(y_test, y_pred_test_threshold))

0.7368421052631579


In [109]:
print(fbeta_score(y_test, y_pred_test_threshold, beta=2))

0.43478260869565216
